In [ ]:
import kagglehub

# Download dataset
data_path = kagglehub.dataset_download("atulyakumar98/gundetection")

# print("Path to dataset files:", data_path)

In [ ]:
import os
import glob
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models

# Get all image paths
all_images = sorted(glob.glob(os.path.join(data_path, "*.jpg")))

# Extract labels from txt files
labels_list = []
for img_file in all_images:
    txt_file = img_file.replace(".jpg", ".txt")
    if os.path.exists(txt_file) and os.path.getsize(txt_file) > 0:
        labels_list.append(1)  # 1 = Gun detected
    else:
        labels_list.append(0)  # 0 = No Gun / Background

labels_array = np.array(labels_list, dtype=np.int32)
print(f"Total images: {len(all_images)} | Guns: {sum(labels_array)} | No Guns: {len(labels_array) - sum(labels_array)}")

In [ ]:
# Train/Test split (75/25)
X_train, X_test, y_train, y_test = train_test_split(
    all_images, labels_array, test_size=0.25, random_state=101, stratify=labels_array
)

In [ ]:
IMG_H = 224
IMG_W = 224
BATCH = 64

def load_and_preprocess(filepath, label):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_H, IMG_W])
    img = img / 255.0
    return img, label

# Training dataset
train_data = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_data = train_data.shuffle(len(X_train)).map(load_and_preprocess).batch(BATCH).prefetch(tf.data.AUTOTUNE)

# Validation dataset
test_data = tf.data.Dataset.from_tensor_slices((X_test, y_test))
test_data = test_data.map(load_and_preprocess).batch(BATCH).prefetch(tf.data.AUTOTUNE)

In [ ]:
# Build CNN Architecture with BatchNormalization
gun_model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(256, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(1, activation='sigmoid')
])

# Compile with RMSprop optimizer (different from adam)
gun_model.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.0005),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

gun_model.summary()

In [ ]:
# Early stopping callback
es_callback = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)

# Train the model
results = gun_model.fit(
    train_data,
    validation_data=test_data,
    epochs=15,
    callbacks=[es_callback]
)